# 02 - Elbow Deteministic Forecasts

In [1]:
import sys
from pathlib import Path
# add project root (parent of notebook folder) to path
sys.path.append(str(Path("..").resolve()))
from verification_plots import *

In [2]:
from pathlib import Path
from veriflow import run_pipeline
from veriflow.constants import VERSION_FULL
import logging
from dotenv import load_dotenv
import warnings

# Reload automatically
%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

load_dotenv(dotenv_path="tutorial.env", override=True)

base_config = Path("config")
base_config.exists()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logging.info(f"Running Veriflow version {VERSION_FULL}")

2026-06-12 17:11:41,403 - INFO - Running Veriflow version 0.0.3+ce113c38b205ef314b60ecf3c7f3a7cc54ca5510.dirty


# Deterministic Precipitation

In [25]:
## NWP Precipitation
config_files = [
    "Elbow_GDPS_Precip.yaml",
]

ods = run_pipeline(config=(Path(base_config, config_files[0]), "yaml"))


2026-06-12 20:30:33,089 - INFO - Successfully initialized the configuration. 
	 verification_period_start = 2026-05-15 00:00:00 
	 verification_period_end = 2026-06-01 00:00:00
2026-06-12 20:30:33,092 - INFO - Start getting data from FewsWebservice.
2026-06-12 20:30:33,589 - INFO - Successfully got data from FewsWebservice.
2026-06-12 20:30:33,590 - INFO - Start getting data from FewsWebservice.
2026-06-12 20:30:34,143 - INFO - Successfully got data from FewsWebservice.
2026-06-12 20:30:34,144 - INFO - Successfully loaded all data from sources.
2026-06-12 20:30:34,164 - INFO - Successfully computed ContinuousScores for verification pair PC.
2026-06-12 20:30:34,213 - INFO - Successfully wrote results of verification pair PC to CFCompliantNetCDF.
2026-06-12 20:30:34,214 - INFO - Verification pipeline completed successfully.


# Evaluation of results

Verification metrics and results can contain a level of abstraction. Although these abstractions can reveal important information about forecast quality, a basic "eyeball verification" is often the best and intuitive way to start your verification exercise. You'll likely find strengths and weaknesses in your forecasts early on, without directly diving into levels of abstraction. In addition, a solid visual inspection may help you later on in understanding or explaining the more abstract results.

## 1 - Visual inspection of observed and forecast data
A good starting point for "eyeball" verification is simple: just looking at your observations and forecasts in a visual way. Use the interactive elements in the plots below to zoom, pan and compare the results of our 3 NWP products.

In [27]:
stations = ods.get(ods.verification_pairs[0]).coords["station"].values

lead_times = ods.get(ods.verification_pairs[0]).coords["lead_time"].values
lead_times_hours = [lt.astype('timedelta64[h]').astype(int) for lt in lead_times]

print(f'Stations: {stations}')

print(f'GDPS Lead times (hours): {lead_times_hours}')


Stations: ['3031092' '3050778' 'MSC-005' '05BL813' '05BJ804' '05BL809' 'FIRES-B4'
 '05BJ805' '05BL812' 'FIRES-B5' '05BJ806' '05BH803' '05BF825' '05BH802'
 '05BL810' '05BF827']
GDPS Lead times (hours): [np.int64(24), np.int64(48), np.int64(72), np.int64(96), np.int64(120), np.int64(144), np.int64(168)]


In [ ]:
from verification_plots import forecast_timeseries_plot

forecast_timeseries_plot(ods, station=stations[1])


## 2 - Looking into the Mean Error per lead time


In [32]:
from verification_plots import crps_plot
crps_plot(ods, score_var="mean_error")